In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion

# load R package
ro.r('.libPaths(c("/dcs/23/u2200504/R/x86_64-redhat-linux-gnu-library/4.5", .libPaths()))')
ro.r('library(dagbagM)')

#setup project root and paths
project_root = Path("/dcs/23/u2200504/thesis/recidivism-causal").resolve()

#set NIJ root
nij_root = project_root / "data" / "processed"

#path to put results
output_dir = project_root/"results"/"NIJ"/"graphs_DAGBagM"
output_dir.mkdir(parents=True,exist_ok=True)

#test it works
print("nij root:" , nij_root)
print("Output directory:", output_dir)

nij root: /dcs/23/u2200504/thesis/recidivism-causal/data/processed
Output directory: /dcs/23/u2200504/thesis/recidivism-causal/results/NIJ/graphs_DAGBagM


In [2]:
#assigns node types. 'c' for continuous, 'b' for binary
def infer_type(df):
    import rpy2.robjects as ro
    node_types=[]
    for col in df.columns:
        x=df[col].dropna()
        unique_vals=x.unique()
        if np.issubdtype(x.dtype, np.number):
            if len(unique_vals) == 2 and set(unique_vals).issubset({0, 1}):
                node_types.append("b")
            else:
                node_types.append("c")
        else:
            if len(unique_vals) == 2:
                node_types.append("b")
            else:
                node_types.append("c")
    return ro.StrVector(node_types)

def encode_mixed_df(df):
    df_enc = df.copy()
    for col in df_enc.columns:
        #use categorical code for non numeric
        if not np.issubdtype(df_enc[col].dtype, np.number):
            df_enc[col] = df_enc[col].astype("category").cat.codes
    return df_enc

In [3]:
def run_dagbagm(df: pd.DataFrame, seed=1):
    df_clean = df.dropna().copy()
    df_clean = encode_mixed_df(df_clean)
    #infer node types on transformed df
    node_type = infer_type(df_clean)

    print("Columns used in dagbagM:", df_clean.columns.tolist())
    print("dtypes:", df_clean.dtypes)

    start = time.time()
    with (ro.default_converter + pandas2ri.converter).context():
        #Python ->R
        Y_r = conversion.py2rpy(df_clean)

        ro.globalenv["Y"] = Y_r
        ro.globalenv["node_type"] = node_type
        ro.globalenv["seed"] = seed
        ro.r('''
        set.seed(seed)
        Y_df <- as.data.frame(Y)

        # Convert to numeric matrix for hc
        Y_mat <- as.matrix(Y_df)
        
        temp <- dagbagM::hc(
          Y = Y_mat,
          nodeType = node_type,
          whiteList = NULL,
          blackList = NULL,
          tol = 1e-6,
          standardize = FALSE,
          maxStep = 1000,
          restart = 10,
          verbose = FALSE
        )
        adj_mat <- temp$adjacency
        col_names <- colnames(Y_mat)
        ''')
        #R -> Python
        adjacency = conversion.rpy2py(ro.r('adj_mat'))
        col_names= list(ro.r('col_names'))

    end = time.time()
    print(f"DAGBagM took {(end - start)/60:.2f} minutes")
    return np.asarray(adjacency), col_names #list(df_clean.columns) #returns labels.

In [4]:
import networkx as nx
import matplotlib.pyplot as plt

def draw_graph(adj, nodes, output_path):
    G = nx.DiGraph()
    #add nodes
    G.add_nodes_from(nodes)
    
    #add directed edges, adj[i,j] = 1 => i -> j
    for i, src in enumerate(nodes):
        for j, tgt in enumerate(nodes):
            if adj[i, j] == 1:
                G.add_edge(src, tgt)

    plt.figure(figsize=(10,8))
    pos = nx.spring_layout(G, k=1.2, iterations=500, seed=0)
    nx.draw(G, pos, with_labels=True, labels={node: node for node in nodes},
            node_size=900, font_size=8, arrowsize=10)
    plt.savefig(out_path, dpi=150)
    plt.close()

In [5]:
from graphviz import Digraph
from pathlib import Path
import numpy as np

def draw_graphviz_dag(adj, out_path, node_labels=None, engine="dot"):

    adj = np.asarray(adj)
    if adj.ndim == 1:
        adj = adj.reshape(1, 1)

    n = adj.shape[0]

    # Default labels
    if node_labels is None:
        node_labels = [f"X{i}" for i in range(n)]
    else:
        node_labels = list(node_labels)[:n]

    out_path = Path(out_path)
    g = Digraph(format="png", engine=engine)

    # Thesis‑friendly black & white style
    g.attr(rankdir="TB")  # top‑to‑bottom; use "LR" if you prefer left‑to‑right
    g.attr(
        "node",
        shape="ellipse",
        style="solid",
        color="black",
        fontname="Helvetica",   # or "Times New Roman" / "Palatino"
        fontsize="10",
    )
    g.attr(
        "edge",
        color="black",
        arrowsize="0.7",
    )

    # Add nodes
    for name in node_labels:
        g.node(name, label=name)

    # Add edges for weights above threshold
    for i, src in enumerate(node_labels):
        for j, tgt in enumerate(node_labels):
            w = adj[i, j]
            if abs(w) > 0:
                g.edge(src, tgt)

    out_path.parent.mkdir(parents=True, exist_ok=True)
    g.render(filename=out_path.with_suffix("").as_posix(), cleanup=True)


In [6]:
import time
csv_path = nij_root/"NIJ_lean_compact_onehot.csv"
df = pd.read_csv(csv_path)

# Run DAGBagM on this dataset
adj,nodes = run_dagbagm(df, seed=1)

out_path = output_dir / "NIJ_graph_DAGBagM"

#use graphviz instead
draw_graphviz_dag(adj, out_path, node_labels=nodes)

Columns used in dagbagM: ['Gender_F', 'Race_BLACK', 'Age_at_Release_18-22', 'Age_at_Release_23-27', 'Age_at_Release_28-32', 'Age_at_Release_33-37', 'Age_at_Release_38-42', 'Age_at_Release_43-47', 'Age_at_Release_48 or older', 'Gang_Affiliated', 'Prior_Arrest_Episodes_Felony_high', 'Prior_Arrest_Episodes_Property_high', 'Prior_Arrest_Episodes_Drug_high', 'Prior_Conviction_Episodes_Felony_high', 'Percent_Days_Employed', 'Jobs_Per_Year', 'Avg_Days_per_DrugTest', 'DrugTests_Cocaine_Positive', 'Delinquency_Reports_high', 'Supervision_Risk_Score_First', 'Recidivism_Within_3years']
dtypes: Gender_F                                    int8
Race_BLACK                                  int8
Age_at_Release_18-22                        int8
Age_at_Release_23-27                        int8
Age_at_Release_28-32                        int8
Age_at_Release_33-37                        int8
Age_at_Release_38-42                        int8
Age_at_Release_43-47                        int8
Age_at_Release_48 

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 2:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  algorithm did not converge
  
R callback write-console: 3:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   
R callback write-console:  the line search routine failed, possibly due to insufficient numeric precision
  
R callback write-console: 4:   
R callback write-console: In hc_(Y, nodeType, whiteList, blackList, tol, maxStep, restart,  :  
R callback write-console: 
   


DAGBagM took 6.07 minutes


TypeError: draw_graphviz_dag() got an unexpected keyword argument 'threshold'

In [7]:
#use graphviz instead
draw_graphviz_dag(adj, out_path, node_labels=nodes)

In [9]:
G = nx.DiGraph()
#add nodes
G.add_nodes_from(nodes)

#add directed edges, adj[i,j] = 1 => i -> j
for i, src in enumerate(nodes):
    for j, tgt in enumerate(nodes):
        if adj[i, j] == 1:
            G.add_edge(src, tgt)

In [19]:
import networkx as nx
from collections import Counter

AGE_PREFIX = "Age_at_Release_"          # prefix of one-hot age bucket columns
AGE_MERGED = "Age_at_Release"           # name of merged age node
TARGET     = "Recidivism_Within_3years" #target node
#G assumed to be an nx.DiGraph()

#Merge age buckets
age_nodes = [n for n in G.nodes if isinstance(n, str) and n.startswith(AGE_PREFIX)]
G_merged = nx.DiGraph()

# copy all nodes except age buckets
for n in G.nodes:
    if n not in age_nodes:
        G_merged.add_node(n)

# add merged age node if any buckets exist
if age_nodes:
    G_merged.add_node(AGE_MERGED)

age_out = Counter()   #counts outgoing edges from age buckets Age -> X (bucket -> non-age)
age_in  = Counter()   #counts incoming edges ".. "(non-age -> bucket)

for u, v, data in G.edges(data=True):

    #ignore edges between age buckets
    if u in age_nodes and v in age_nodes:
        continue

    #edges not affecting age buckets are just copied over
    if u not in age_nodes and v not in age_nodes:
        G_merged.add_edge(u, v, **data)
        continue

    #bucket -> non-age variable
    if u in age_nodes and v not in age_nodes:
        age_out[v] += 1

    #non-age variable -> bucket
    if v in age_nodes and u not in age_nodes:
        age_in[u] += 1

#decide majority direction for each neighbour of an age bucket
for x, cnt_out in age_out.items():
    cnt_in = age_in.get(x, 0)

    if cnt_out > cnt_in:
        # majority oriented as Age bucket -> X
        G_merged.add_edge(AGE_MERGED, x)
    elif cnt_in > cnt_out:
        # majority X -> Age bucket
        G_merged.add_edge(x, AGE_MERGED)
    else:
        #tie: drop, dont add edge to age merged
        pass

# add edges X -> Age for neighbours that only ever had incoming-to-bucket edges
for x, cnt_in in age_in.items():
    if x in age_out:
        continue    # already handled above
    G_merged.add_edge(x, AGE_MERGED)

if TARGET not in G_merged:
    raise ValueError(f"Target node '{TARGET}' not found in graph; "
                     f"available nodes include: {list(G_merged.nodes)[:10]} ...")

parents  = set(G_merged.predecessors(TARGET))
markov_blanket_nodes = parents | {TARGET}

G_mb = G_merged.subgraph(markov_blanket_nodes).copy()

print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))


Original nodes: 21
After age-merge: 15
Markov blanket nodes: 9


In [18]:
node_labels = list(G_mb.nodes())          
adj = nx.to_numpy_array(G_mb, nodelist=node_labels, dtype=int)

out_path = output_dir/"NIJ_graph_DAGBagM_PRUNED"
draw_graphviz_dag(adj, out_path, node_labels=node_labels, engine="dot")

In [21]:
nodes = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodes, dtype=int)
adj_df = pd.DataFrame(A, index=nodes, columns=nodes)
adj_df.to_csv(output_dir/ "NIJ_graph_DAGBagM_adj.csv", index=True)

In [24]:
print("Original edges:", len(G.edges()))
print("After age-merge:", len(G_merged.edges()))
print("Markov blanket edges:", len(G_mb.edges()))


Original edges: 132
After age-merge: 74
Markov blanket edges: 29


In [25]:
print("Original nodes:", len(G.nodes))
print("After age-merge:", len(G_merged.nodes))
print("Markov blanket nodes:", len(G_mb.nodes))


Original nodes: 21
After age-merge: 15
Markov blanket nodes: 9
